# What the crystal is made of

The dose depends on how strongly the crystal absorbs, which depends on its
atoms. RADDOSE-3D offers several ways to say what those are -- from a unit
cell and residue counts, a sequence, a PDB or CIF file, a solution
concentration -- and each becomes the same thing in the end: absorption,
scattering and attenuation coefficients at the beam energy, and a density.

In [ ]:
import matplotlib.pyplot as plt

from pyraddose_3d.models.experiment import Experiment
from pyraddose_3d.solve import simulate

DATA = "../../tests/data"
INSULIN_CELL = [78.02, 78.02, 78.02]


def experiment(composition):
    """The same crystal, beam and wedge, with a given composition."""
    return Experiment.model_validate(
        {
            "crystal": {
                "type": "cuboid",
                "dimensions": [40.0, 30.0, 20.0],
                "pixels_per_micron": 0.5,
                "composition": composition,
            },
            "beam": {
                "type": "gaussian",
                "flux": 2e12,
                "energy": 12.1,
                "fwhm": [15.0, 15.0],
            },
            "wedges": [{"start": 0.0, "end": 90.0, "exposure_time": 30.0}],
        }
    )

## The routes

Each is one `method`. The keyword RADDOSE-3D's input files use is in the
comment.

In [ ]:
compositions = {
    # AbsCoefCalc Average: fixed coefficients for a nominal protein crystal.
    "average": {"method": "average"},
    # AbsCoefCalc RD3D: unit cell contents counted by hand.
    "unit cell": {
        "method": "unit_cell",
        "unit_cell": INSULIN_CELL,
        "num_monomers": 24,
        "num_residues": 51,
        "protein_heavy_atoms": {"Zn": 0.333, "S": 6},
        "solvent_heavy_conc": {"P": 425},
    },
    # AbsCoefCalc Sequence: residues, and their sulphur, from a FASTA file.
    "sequence": {
        "method": "sequence",
        "unit_cell": INSULIN_CELL,
        "num_monomers": 24,
        "sequence_file": "data/insulin.fasta",
        "protein_heavy_atoms": {"Zn": 0.333},
        "solvent_heavy_conc": {"P": 425},
    },
    # AbsCoefCalc EXP: atoms read from a PDB file (here insulin, 3I40).
    "PDB": {
        "method": "pdb",
        "pdb": f"{DATA}/3I40.pdb",
        "solvent_heavy_conc": {"P": 425},
    },
    # AbsCoefCalc SAXS: a protein solution in mg/ml -- a sample in a
    # capillary rather than a crystal, here for completeness.
    "SAXS": {
        "method": "saxs",
        "unit_cell": [100.0, 100.0, 100.0],
        "protein_concentration": 10.0,
        "num_residues": 129,
    },
    # AbsCoefCalc SAXSSEQ: the same, with the molecule read from a FASTA
    # file. With no unit cell, the box of solution is 1000 A on a side.
    "SAXS sequence": {
        "method": "saxs_sequence",
        "protein_concentration": 10.0,
        "sequence_file": "data/insulin.fasta",
    },
    # AbsCoefCalc SmallMole: a small-molecule crystal from its formula.
    "small molecule": {
        "method": "small_molecule",
        "unit_cell": [10.012, 18.808, 6.334],
        "num_monomers": 8,
        "atoms": {"Mg": 1, "O": 3},
    },
    # AbsCoefCalc EXPSM: the formula read from a CIF (here antimony-bearing).
    "CIF": {"method": "cif", "cif": f"{DATA}/COD-1008123.cif"},
}

runs = {name: simulate(experiment(c)) for name, c in compositions.items()}

print(f"{'':15s} {'absorption':>11} {'attenuation':>12} {'density':>8} {'max dose':>9}")
print(f"{'':15s} {'(1/um)':>11} {'(1/um)':>12} {'(g/ml)':>8} {'(MGy)':>9}")
for name, simulation in runs.items():
    c = simulation.coefficients
    density = float(simulation.density)
    max_dose = float(simulation.summaries[0].max_dose)
    print(
        f"{name:15s} {float(c.absorption):11.3e} {float(c.attenuation):12.3e} "
        f"{density:8.2f} {max_dose:9.1f}"
    )

The unit cell, the sequence and the PDB file all describe insulin.
Counting residues by hand and reading them from the sequence agree closely.
The PDB route takes the cell from the file's CRYST1 line, the residues from
SEQRES and any bound heavy atoms from HETATM; 3I40 is insulin in a slightly
smaller cell, 77.22 rather than 78.02 Angstroms, and models no zinc, so it
absorbs a little less. The average coefficients are a generic protein, not
this one.

The two SAXS rows are solutions at 10 mg/ml, so they are mostly water.
That is why they agree so closely, although one counts 129 residues and
the other reads insulin's sequence.

The antimony compound absorbs about forty times more strongly than any of
them. Absorption rises steeply with atomic number, so a few heavy atoms
dominate it.

## Solvent

For the unit-cell route, RADDOSE-3D computes the solvent fraction from
the cell's contents if it is not given. Giving it changes how much of the
cell is water rather than protein -- but water and protein absorb much
alike, so the dose barely moves:

In [ ]:
fractions = [0.3, 0.4, 0.5, 0.6, 0.7]
doses = []
for fraction in fractions:
    unit_cell = {**compositions["unit cell"], "solvent_fraction": fraction}
    doses.append(float(simulate(experiment(unit_cell)).summaries[0].max_dose))
computed = float(runs["unit cell"].summaries[0].max_dose)

fig, ax = plt.subplots(figsize=(5.5, 3.5), constrained_layout=True)
ax.plot(fractions, doses, "o-", label="solvent fraction given")
ax.axhline(computed, color="grey", ls="--", label="computed from the cell")
ax.set_xlabel("solvent fraction")
ax.set_ylabel("max dose (MGy)")
ax.legend()
plt.show()

## Heavy atoms in the solvent

What does move it is heavy atoms. Those in the solvent -- salts,
cryoprotectant additives, soaks -- are given in mmol/l; here, increasing
amounts of selenium:

In [ ]:
for millimolar in (0, 100, 500, 1000):
    solvent = {"P": 425, "Se": millimolar} if millimolar else {"P": 425}
    unit_cell = {**compositions["unit cell"], "solvent_heavy_conc": solvent}
    run = simulate(experiment(unit_cell))
    absorption = float(run.coefficients.absorption)
    print(
        f"Se {millimolar:4d} mM: absorption {absorption:.3e} /um, "
        f"max dose {float(run.summaries[0].max_dose):6.1f} MGy"
    )